# **NBC para clasificación de péptidos antimicrobianos**

Profesor: Dr. Irvin Hussein López-Nava

Materia: Reconocimiento de Patrones

Alumna: Jazmín Alejandra Martínez Guerrero

## **Obtención de los datos**

La muestra positiva (los péptidos antimicrobianos) se obtuvieron de https://aps.unmc.edu/downloads. 
Se utilizaron los que dicen *2024 natural antimicrobial peptides (AMPs) with known activity (3306 entries)*.

La muestra negativa (los péptidos no antimibrobianos) la obtuve de https://www.uniprot.org/uniprotkb
Y filtré aquellos que tenían de 1-200 aminoácidos (180,269)


## **Lectura de los datos**


**Datos positivos**

In [25]:
# Función para leer los datos positivos

def parse_fasta_positive(file_path):
    sequences = {}
    with open(file_path, 'r') as file:
        current_id = ""
        for line in file:
            line = line.strip()
            if line.startswith(">"):
                current_id = line[1:]
                if current_id.startswith("Your search"):
                    current_id = ""
                    continue
                sequences[current_id] = ""
            elif current_id:
                sequences[current_id] += line
    return sequences

In [29]:
positives_raw = parse_fasta_positive("positive.txt")
print(type(positives_raw))
print(positives_raw)
print(len(positives_raw))

<class 'dict'>
{'AP00001': 'GLWSKIKEVGKEAAKAAAKAAGKAALGAVSEAV', 'AP00002': 'YVPLPNVPQPGRRPFPTFPGQGPFNPKIKWPQGY', 'AP00003': 'DGVKLCDVPSGTWSGHCGSSSKCSQQCKDREHFAYGGACHYQFPSVKCFCKRQC', 'AP00004': 'NLCERASLTWTGNCGNTGHCDTQCRNWESAKHGACHKRGNWKCFCYFDC', 'AP00005': 'VFIDILDKVENAIHNAAQVGIGFAKPFEKLINPK', 'AP00006': 'GNNRPVYIPQPRPPHPRI', 'AP00007': 'GNNRPVYIPQPRPPHPRL', 'AP00008': 'RLCRIVVIRVCR', 'AP00009': 'RFRPPIRRPPIRPPFYPPFRPPIRPPIFPPIRPPFRPPLGPFP', 'AP00010': 'RRIRPRPPRLPRPRPRPLPFPRPGPRPIPRPLPFPRPGPRPIPRPLPFPRPGPRPIPRPL', 'AP00011': 'WNPFKELERAGQRVRDAVISAAPAVATVGQAAAIARG', 'AP00012': 'GLFDIIKKIAESI', 'AP00013': 'GLFDIIKKIAESF', 'AP00014': 'GLLDIVKKVVGAFGSL', 'AP00015': 'GLFDIVKKVVGALGSL', 'AP00016': 'GLFDIVKKVVGAIGSL', 'AP00017': 'GLFDIVKKVVGTLAGL', 'AP00018': 'GLFDIVKKVVGAFGSL', 'AP00019': 'GLFDIAKKVIGVIGSL', 'AP00020': 'GLFDIVKKIAGHIAGSI', 'AP00021': 'GLFDIVKKIAGHIASSI', 'AP00022': 'GLFDIVKKIAGHIVSSI', 'AP00023': 'AACARFIDDFCDTLTPNIYRPRDNGQRCYAVNGHRCDFTVFNTNNGGNPIRASTPNCKTVLRTAANRCPTGGRGKIN

**Datos negativos**

In [27]:
def parse_fasta_negative(filepath):
    sequences = {}
    
    with open(filepath, 'r') as file:
        content = file.read()
        
    blocks = content.strip().split('>')
    
    for block in blocks:
        if not block:
            continue
            
        lines = block.splitlines()
        header = lines[0]
        
        seq_id = header.split('|')[1] if '|' in header else header.split()[0]
        sequence = "".join(lines[1:]).strip()
        
        if sequence:
            sequences[seq_id] = sequence
            
    return sequences

In [34]:
negatives_raw = parse_fasta_negative("negative.fasta")
print(type(negatives_raw))
print(len(negatives_raw))

<class 'dict'>
180264


## **Filtrado de datos**

Se tienen que filtrar los datos de la muestra negativa porque son muchos más (180264) que los positivos (3306), además tenemos que fijar que el número de secuencias sean parecidas entre ambas bases (sino al algoritmo podría clasificar con base a la longitud de la secuencia)

Primero debemos de asegurarnos de que los peptidos tengan toda su secuencia de aminoácidos, porque puede suceder que aparezcan X (lo cual significa que el aminoácido es desconocido) y si tenemos un aminoácido desconcido, no podemos calcular las características del péptido

In [83]:
print(len(positives_raw))
print(len(negatives_raw))

3306
180264


Tenemos 3306 datos positivos y 180264 negativos

In [84]:
def filter_canonical_peptides(peptide_dict):
    valid_aas = set("ACDEFGHIKLMNPQRSTVWY")
    return {k: v for k, v in peptide_dict.items() if set(v.upper()).issubset(valid_aas)}

positives_raw = filter_canonical_peptides(positives_raw)
negatives_raw = filter_canonical_peptides(negatives_raw)
print(len(positives_raw))
print(len(negatives_raw))

3306
179073


Tenemos 3306 datos positivos y 179073 negativos, todos con sus aminoácidos conocidos

In [85]:
# Para conocer la longitud de las secuencias de nuestras bases vamos a ver los cuartines de longitud de la cantidad de aminoacidos por peptido
import statistics

def get_peptide_length_quartiles(peptide_dict):
    lengths = [len(seq) for seq in peptide_dict.values()]
    
    if not lengths:
        return {}
        
    if len(lengths) < 4:
        raise ValueError("Insufficient data")
        
    quartiles = statistics.quantiles(lengths, n=4)

    return {
        'Q1': quartiles[0],
        'Q2': quartiles[1],
        'Q3': quartiles[2],
        'Min': min(lengths),
        'Max' : max(lengths)
    }

In [86]:
print(f"Positives {get_peptide_length_quartiles(positives_raw)}")
print(f"Negatives {get_peptide_length_quartiles(negatives_raw)}")

Positives {'Q1': 20.0, 'Q2': 29.0, 'Q3': 40.0, 'Min': 2, 'Max': 183}
Negatives {'Q1': 89.0, 'Q2': 127.0, 'Q3': 161.0, 'Min': 2, 'Max': 200}


Se filtran los negativos con un muestreo que ayude a tener una distribución de longitud similar a la de los positivos.

In [87]:
# Esta función cuenta cuántos aminoácidos tiene cada peptido en positivos, 
# y luego muestrea los negativos con longitudes iguales para que la distribución sea la misma
# En el muestro NO se permite que haya intersección entre positivos y negativos
# Para esto primero se excluyen los positivos que están en el conjunto de negativos
# Para asegurarnos de que haya la misma cantidad en ambos conjuntos, dejamos deficit

import random
from collections import defaultdict

def match_peptide_distribution(positives_dict, negatives_dict):
    pos_sequences = set(positives_dict.values())
    
    target_distribution = defaultdict(int)
    for seq in positives_dict.values():
        target_distribution[len(seq)] += 1
        
    negatives_by_length = defaultdict(list)
    seen_sequences = set()
    
    for k, v in negatives_dict.items():
        if v not in seen_sequences and v not in pos_sequences:
            seen_sequences.add(v)
            negatives_by_length[len(v)].append((k, v))
            
    sampled_negatives = {}
    unused_negatives = []
    deficit = 0
    
    for length, target_count in target_distribution.items():
        available = negatives_by_length.get(length, [])
        
        if target_count <= len(available):
            sampled_items = random.sample(available, target_count)
            sampled_negatives.update(sampled_items)
            
            sampled_keys = set(k for k, v in sampled_items)
            unused_negatives.extend([(k, v) for k, v in available if k not in sampled_keys])
        else:
            sampled_negatives.update(available)
            deficit += target_count - len(available)
            
    if deficit > 0:
        if deficit <= len(unused_negatives):
            extra_samples = random.sample(unused_negatives, deficit)
            sampled_negatives.update(extra_samples)
        else:
            sampled_negatives.update(unused_negatives)
            
    return sampled_negatives

In [88]:
random.seed(26)
negatives_filtered = match_peptide_distribution(positives_raw, negatives_raw)

print(f"Positives {get_peptide_length_quartiles(positives_raw)}")
print(f"Negatives {get_peptide_length_quartiles(negatives_filtered)}")

print(len(positives_raw))
print(len(negatives_filtered))

Positives {'Q1': 20.0, 'Q2': 29.0, 'Q3': 40.0, 'Min': 2, 'Max': 183}
Negatives {'Q1': 20.0, 'Q2': 29.0, 'Q3': 40.0, 'Min': 2, 'Max': 183}
3306
3306


Se agregan descriptores para los peptidos, los cuales usaremos para el modelo de clasificación, es decir, agregamos las variables.
Para esto utilizaré la función calculate_all() los detalles se pueden consultar en https://modlamp.org/modlamp.html


In [78]:
#pip install modlamp

In [89]:
from modlamp.descriptors import GlobalDescriptor

def calculate_global_descriptors(peptide_dict):
    peptide_ids = list(peptide_dict.keys())
    sequences = list(peptide_dict.values())
    
    descriptor_instance = GlobalDescriptor(sequences)
    descriptor_instance.calculate_all()
    
    features_dict = {}
    for i, seq_id in enumerate(peptide_ids):
        features_dict[seq_id] = descriptor_instance.descriptor[i]
        
    return features_dict

In [96]:
positives_descriptors = calculate_global_descriptors(positives_raw)
negatives_descriptors = calculate_global_descriptors(negatives_filtered)

# Se combinan los diccionarios para tener toda la información junta 
def combine_peptide_data(sequence_dict, descriptor_dict):
    combined = {}
    for k, v in sequence_dict.items():
        if k in descriptor_dict:
            combined[k] = {
                'sequence': v,
                'features': descriptor_dict[k]
            }
    return combined

positives_complete = combine_peptide_data(positives_raw, positives_descriptors)
negatives_complete = combine_peptide_data(negatives_filtered, negatives_descriptors)

Se dividen los datos en train y test

In [97]:
def create_train_test_splits(positives, negatives):
    def split_dict(data):
        keys = list(data.keys())
        random.shuffle(keys)
        
        split_idx = int(len(keys) * 0.8)
        
        train = {k: data[k] for k in keys[:split_idx]}
        test = {k: data[k] for k in keys[split_idx:]}
        
        return train, test

    train_positives, test_positives = split_dict(positives)
    train_negatives, test_negatives = split_dict(negatives)
    
    return train_positives, train_negatives, test_positives, test_negatives

train_positives, train_negatives, test_positives, test_negatives = create_train_test_splits(positives_complete, negatives_complete)

In [98]:
print(len(train_positives))
print(len(train_negatives))
print(len(test_positives))
print(len(test_negatives))

2644
2644
662
662


In [99]:
train_positives

{'AP02323': {'sequence': 'KRKKHRCRVYNNGMPTGMYRWC',
  'features': array([2.20000000e+01, 2.78433000e+03, 7.71800000e+00, 2.77194154e-03,
         1.13632812e+01, 5.93318182e+01, 1.36363636e-01, 1.31818182e+01,
         3.70954545e+00, 2.27272727e-01])},
 'AP02232': {'sequence': 'AATAKKGAKKADAPAKPKKATKPKSPKKAAKKAGAKKGVKRAGKKGAKKTTKAKK',
  'features': array([5.50000000e+01, 5.56681000e+03, 2.39770000e+01, 4.30713461e-03,
         1.22519531e+01, 1.51236364e+01, 0.00000000e+00, 3.25454545e+01,
         2.34690909e+00, 2.90909091e-01])},
 'AP01680': {'sequence': 'ILGIITSLLKSLGKK',
  'features': array([ 1.50000000e+01,  1.58302000e+03,  3.98800000e+00,  2.51923539e-03,
          1.30781250e+01,  5.19333333e+00,  0.00000000e+00,  1.82000000e+02,
         -6.86666667e-01,  4.66666667e-01])},
 'AP00237': {'sequence': 'KSCCPTTTARNIYNTCRFGGGSRPVCAKLSGCKIISGTKCDSGWNH',
  'features': array([4.60000000e+01, 4.87961000e+03, 6.10400000e+00, 1.25091964e-03,
         9.60644531e+00, 5.75739130e+01, 6.52